# 01. Khám phá dữ liệu (EDA)

Phân bố nhãn, độ dài caption, đặc điểm ảnh, teencode/emoji.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
import pandas as pd
from collections import Counter
from PIL import Image
from vimmsd.data.dataset import load_all_records

cfg = load_config("configs/base.yaml")
splits = load_all_records(cfg)
df = pd.DataFrame([{**r, "split": s} for s, recs in splits.items() for r in recs])
labels = list(cfg.data.labels)
df["label_name"] = df["label"].map(lambda i: labels[i] if i >= 0 else "(không nhãn)")
df.groupby("split").size()

## Phân bố nhãn

In [ ]:
labeled = df[df.split != "public_test"]
counts = labeled["label_name"].value_counts().reindex(labels)
display(pd.DataFrame({"số mẫu": counts, "tỉ lệ": (counts / counts.sum()).round(4)}))
ax = counts.plot.bar(figsize=(6, 3.5), rot=0, color="#4c72b0")
ax.set_yscale("log"); ax.set_ylabel("số mẫu (log)"); ax.set_title("Phân bố nhãn (train có nhãn)")
plt.tight_layout(); plt.savefig("reports/figures/eda_label_distribution.png", dpi=150); plt.show()

## Độ dài caption

In [ ]:
labeled = labeled.assign(n_words=labeled.caption.str.split().str.len())
display(labeled.groupby("label_name").n_words.describe().reindex(labels))
labeled.boxplot(column="n_words", by="label_name", figsize=(7, 4), showfliers=False)
plt.suptitle(""); plt.title("Số từ trong caption theo nhãn"); plt.show()

## Emoji, hashtag, teencode

In [ ]:
import emoji, re
from vimmsd.data.preprocessing import TEENCODE, WORD_RE

labeled = labeled.assign(
    n_emoji=labeled.caption.map(emoji.emoji_count),
    has_hashtag=labeled.caption.str.contains("#"),
    n_teencode=labeled.caption.map(lambda t: sum(w.lower() in TEENCODE for w in WORD_RE.findall(t))),
)
display(labeled.groupby("label_name")[["n_emoji", "has_hashtag", "n_teencode"]].mean().reindex(labels))
top_emoji = Counter(e["emoji"] for t in labeled.caption for e in emoji.emoji_list(t)).most_common(20)
print(top_emoji)

## Tiền xử lý: trước và sau

In [ ]:
from vimmsd.data.preprocessing import TextPreprocessor
pre = TextPreprocessor.from_config(cfg.data.text)
for t in labeled.sample(8, random_state=0).caption:
    print("GỐC :", t)
    print("SAU :", pre(t))
    print()

## Đặc điểm ảnh

In [ ]:
sizes, broken = [], []
for p in df.image_path:
    try:
        with Image.open(p) as im:
            sizes.append((im.size[0], im.size[1], im.format))
    except OSError:
        broken.append(p)
print(f"{len(broken)} ảnh không đọc được (dataset sẽ thay bằng ảnh đen):", broken[:10])
sizes = pd.DataFrame(sizes, columns=["w", "h", "format"]).assign(ratio=lambda d: d.w / d.h)
display(sizes.describe())
display(sizes.format.value_counts())

In [ ]:
labeled = labeled[~labeled.image_path.isin(broken)]
fig, axes = plt.subplots(len(labels), 4, figsize=(14, 3.5 * len(labels)))
for row, name in zip(axes, labels):
    subset = labeled[labeled.label_name == name].sample(4, random_state=1)
    for ax, (_, r) in zip(row, subset.iterrows()):
        ax.imshow(Image.open(r.image_path).convert("RGB")); ax.axis("off")
        ax.set_title(f"[{name}]\n{r.caption[:60]}", fontsize=8)
plt.tight_layout(); plt.show()